# EXP09: ADX 阈值精细扫描 + K-Fold

ADX 是 Keltner 核心入场过滤，当前固定 18。
测试 [12,14,16,18,20,22,25,28,30]，两种点差各一轮 + 冠军 K-Fold

In [ ]:
import sys; sys.path.insert(0, '..')
from backtest import DataBundle, run_variant, run_kfold, C12_KWARGS
from backtest.stats import print_ranked
import pandas as pd

data = DataBundle.load_default()

LIVE_KWARGS = {
    **C12_KWARGS,
    "intraday_adaptive": True,
    "choppy_threshold": 0.35,
    "kc_only_threshold": 0.60,
}
print('Data loaded')

## Part 1: ADX 阈值扫描 ($0.50 spread)

In [ ]:
results_050 = []
for adx in [12, 14, 16, 18, 20, 22, 25, 28, 30]:
    r = run_variant(data, f"ADX={adx}_$0.50",
        **{**LIVE_KWARGS, "keltner_adx_threshold": adx, "spread_cost": 0.50})
    r['adx_th'] = adx
    r['spread'] = 0.50
    results_050.append(r)

print_ranked(results_050)

In [ ]:
# 交易数量 vs ADX 阈值
for r in results_050:
    kc_trades = [t for t in r['_trades'] if t.strategy == 'keltner']
    kc_pnl = sum(t.pnl for t in kc_trades)
    print(f"ADX={r['adx_th']:2d}: {len(kc_trades):4d} KC trades, KC_PnL=${kc_pnl:8.0f}, "
          f"Total Sharpe={r['sharpe']:.2f}")

## Part 2: ADX 阈值扫描 ($0.30 spread)

In [ ]:
results_030 = []
for adx in [12, 14, 16, 18, 20, 22, 25, 28, 30]:
    r = run_variant(data, f"ADX={adx}_$0.30",
        **{**LIVE_KWARGS, "keltner_adx_threshold": adx, "spread_cost": 0.30})
    r['adx_th'] = adx
    r['spread'] = 0.30
    results_030.append(r)

print_ranked(results_030)

In [ ]:
# 两种点差下最优 ADX 对比
best_050 = sorted(results_050, key=lambda r: r['sharpe'], reverse=True)[0]
best_030 = sorted(results_030, key=lambda r: r['sharpe'], reverse=True)[0]
print(f"Best $0.50: ADX={best_050['adx_th']}, Sharpe={best_050['sharpe']:.2f}")
print(f"Best $0.30: ADX={best_030['adx_th']}, Sharpe={best_030['sharpe']:.2f}")
print(f"Same optimal? {best_050['adx_th'] == best_030['adx_th']}")

## Part 3: 冠军 K-Fold 验证

In [ ]:
BEST_ADX = best_050['adx_th']
print(f"Validating ADX={BEST_ADX} with K-Fold ($0.50)")

champ_kwargs = {**LIVE_KWARGS, "keltner_adx_threshold": BEST_ADX, "spread_cost": 0.50}
base_kwargs = {**LIVE_KWARGS, "spread_cost": 0.50}

champ_folds = run_kfold(data, champ_kwargs, label_prefix="Champ_")
base_folds = run_kfold(data, base_kwargs, label_prefix="Base_")

print("\n=== K-Fold Comparison ===")
for b, c in zip(base_folds, champ_folds):
    print(f"{b['fold']}: Base={b['sharpe']:.2f}  ADX{BEST_ADX}={c['sharpe']:.2f}  D={c['sharpe']-b['sharpe']:+.2f}")

wins = sum(1 for b, c in zip(base_folds, champ_folds) if c['sharpe'] > b['sharpe'])
print(f"\nChampion wins {wins}/{len(base_folds)} folds")

In [ ]:
# 参数悬崖：ADX +/-2 扰动
cliff_results = []
for delta in [-4, -2, 0, 2, 4]:
    adx = BEST_ADX + delta
    if adx < 8: continue
    r = run_variant(data, f"ADX={adx}", **{**LIVE_KWARGS, "keltner_adx_threshold": adx, "spread_cost": 0.50})
    cliff_results.append(r)

print_ranked(cliff_results)
sharpes = [r['sharpe'] for r in cliff_results]
print(f"Parameter cliff: {max(sharpes)-min(sharpes):.2f} {'CLIFF!' if max(sharpes)-min(sharpes) > 0.3 else 'smooth'}")

In [ ]:
import json
from backtest.runner import sanitize_for_json
with open('../data/exp09_results.json', 'w') as f:
    json.dump(sanitize_for_json(results_050 + results_030), f, indent=2)
print('Saved')